# Data Preprocessing

This notebook prepares the selected modelling dataset for machine learning. The dataset was generated in `02_Data_Audit_and_Feature_Selection.ipynb` and contains the retained trace element predictor suite.

In [ ]:
# =============================================================================
# Environment setup
# =============================================================================

import os
os.environ["OMP_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.preprocessing import StandardScaler, RobustScaler

ROOT_DIR = Path.cwd().parent

DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORT_DIR = ROOT_DIR / "exports"
FIGURE_DIR = ROOT_DIR / "figures"

EXPORT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 99

print("Environment initialised.")
print(f"Root directory   : {ROOT_DIR}")
print(f"Processed data   : {PROCESSED_DIR}")
print(f"Export directory : {EXPORT_DIR}")
print(f"Figure directory : {FIGURE_DIR}")

## 1. Load Modelling Dataset

Load the fixed modelling dataset created after trace element retention assessment.

In [ ]:
# =============================================================================
# Load modelling dataset and retained elements
# =============================================================================

DATA_FILE = PROCESSED_DIR / "Modelling_Dataset.csv"
ELEMENT_FILE = EXPORT_DIR / "04_retained_trace_elements.csv"

df = pd.read_csv(DATA_FILE, encoding="utf-8")

ELEMENTS = pd.read_csv(
    ELEMENT_FILE,
    encoding="utf-8"
)["Element"].tolist()

META = ["Ref", "Country", "Locality", "Series"]

print("Modelling dataset loaded.\n")

print(f"Analyses   : {len(df)}")
print(f"Elements   : {len(ELEMENTS)}")
print(f"Orogenic   : {(df['Series'] == 'Orogenic').sum()}")
print(f"Anorogenic : {(df['Series'] == 'Anorogenic').sum()}")
print(f"Countries  : {df['Country'].nunique(dropna=True)}")

series_summary = (
    df.groupby("Series")
    .agg(
        Analyses=("Series", "size"),
        Countries=("Country", "nunique")
    )
)

print("\nSeries summary")
display(series_summary)

## 2. Distribution Assessment

Assess the distribution of the retained trace element concentrations to determine whether a logarithmic transformation is appropriate prior to multivariate analysis and machine learning.

In [ ]:
# =============================================================================
# Feature distributions by tectonic series
# =============================================================================

from matplotlib.ticker import ScalarFormatter

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = np.array(axes).flatten()
plot_axes = list(axes[:12]) + [axes[13], axes[14]]
series_colours = {"Orogenic": "tab:blue", "Anorogenic": "tab:red"}

for ax, element in zip(plot_axes, ELEMENTS):
    values = df[element].dropna()
    bins = np.histogram_bin_edges(values, bins="fd")

    for series in ["Orogenic", "Anorogenic"]:
        ax.hist(df.loc[df["Series"].eq(series), element], bins=bins,
                color=series_colours[series], alpha=0.5, edgecolor="black")

    if values.max() >= 1e4:
        formatter = ScalarFormatter(useMathText=True)
        formatter.set_powerlimits((4, 4))
        ax.xaxis.set_major_formatter(formatter)

    ax.set_title(element, fontsize=10)
    ax.grid(alpha=0.3)

axes[12].remove()
axes[15].remove()

legend_handles = [
    Line2D([0], [0], color="tab:blue", lw=6, label="Orogenic"),
    Line2D([0], [0], color="tab:red", lw=6, label="Anorogenic")
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2,
           frameon=False, bbox_to_anchor=(0.5, 0.01))

plt.tight_layout(rect=[0.02, 0.05, 1, 0.95])
outpath = FIGURE_DIR / "06_feature_distributions_raw.png"
plt.savefig(outpath, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {outpath}")

## 3. Log Transformation

Trace element concentrations are transformed using log10 to reduce right skewness and make element distributions more suitable for multivariate analysis.

In [ ]:
# =============================================================================
# Log10 transformation
# =============================================================================

X_raw = df[ELEMENTS].copy()

if (X_raw <= 0).any().any():
    raise ValueError(
        "Log10 transformation cannot be applied because non-positive values are present."
    )

X_log = np.log10(X_raw)

df_log = df[META].copy()
df_log[ELEMENTS] = X_log

print("Log10 transformation applied.")

## 4. Z Score Standardisation

The log transformed trace element concentrations are standardised to zero mean and unit variance for analyses that are sensitive to variable scale, including principal component analysis and support vector machines.

In [ ]:
# =============================================================================
# Z score standardisation
# =============================================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(df_log[ELEMENTS]),
    columns=ELEMENTS,
    index=df_log.index
)

df_scaled = df_log[META].copy()
df_scaled[ELEMENTS] = X_scaled

## 5. Export Processed Datasets

Export the log transformed and log transformed plus Z score standardised datasets for use in subsequent unsupervised and supervised machine learning analyses.

In [ ]:
# =============================================================================
# Export processed datasets
# =============================================================================

outpath = PROCESSED_DIR / "Modelling_Dataset_Log.csv"
df_log.to_csv(outpath, index=False)
print(f"Saved: {outpath}")

outpath = PROCESSED_DIR / "Modelling_Dataset_Log_ZScaled.csv"
df_scaled.to_csv(outpath, index=False)
print(f"Saved: {outpath}")